# Tutorial 6 — Pretrained image classification in PyTorch

Predict top five ImageNet labels with VGG16 and ResNet50, then repeat with AlexNet, ResNet101 and MobileNet. The transfer-learning dataset task is completed in Tutorial 7.

**How to run:** select a Python kernel with the repository requirements, then Run All. QUICK mode keeps experiments short. For the larger run set `TUTORIAL_FULL=1` before starting the kernel. A quick run demonstrates the method; it is not a final accuracy benchmark.

**Execution status:** Complete implementation; end-to-end training/inference has not been verified in this repository run because the required large downloads did not finish. See `VALIDATION.md` for the separate offline model checks. Run All in Colab to generate your own results.

In [ ]:
# Google Colab: fetch the support files and install missing dependencies.
# Local Jupyter: follow README setup once; this cell makes no changes.
import sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    repo = Path("/content/dl-course-tutorials")
    if not repo.exists():
        subprocess.check_call(["git", "clone", "https://github.com/naveed952/Tutorial-01_Perceptron_py.ipynb.git", str(repo)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo/"requirements.txt")])
    import os
    os.chdir(repo)


## Step 1 — Locate the project and set a reproducible seed
A fixed seed makes comparisons easier to reproduce. Each notebook can run independently.

In [ ]:
from pathlib import Path
import sys, os
ROOT = Path.cwd() if (Path.cwd()/"tutorial_support").exists() else Path.cwd().parent
assert (ROOT/"tutorial_support").exists(), "Run from the repository or notebooks folder."
sys.path.insert(0, str(ROOT))
from tutorial_support.common import *
seed()
print("Mode:", "QUICK: small real-data run" if QUICK else "FULL: larger training run")

## Step 2 — Load a sample image and the official model constructors
The task explicitly asks for PyTorch. Each pretrained weight object provides its own preprocessing and category names. Do not apply an arbitrary /255 step after its preprocessing.

In [ ]:
import torch
from torchvision import models
from PIL import Image
import pandas as pd,gc
DEVICE=torch_setup();IMAGE_PATH=sample_image();image=Image.open(IMAGE_PATH).convert('RGB');display(image.resize((320,240)))
architectures=[('VGG16',models.vgg16,models.VGG16_Weights.DEFAULT),('ResNet50',models.resnet50,models.ResNet50_Weights.DEFAULT),('AlexNet',models.alexnet,models.AlexNet_Weights.DEFAULT),('ResNet101',models.resnet101,models.ResNet101_Weights.DEFAULT),('MobileNetV3',models.mobilenet_v3_small,models.MobileNet_V3_Small_Weights.DEFAULT)]

## Step 3 — Run all five models and compare top five probabilities
model.eval() switches off training-only behavior. inference_mode() avoids gradient storage. ImageNet scores only describe the model’s known categories: it cannot reliably name classes absent from ImageNet.

In [ ]:
rows=[]
for name,constructor,weights in architectures:
    model=constructor(weights=weights).to(DEVICE).eval()
    batch=weights.transforms()(image).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():prob=model(batch).softmax(dim=1)[0]
    scores,indices=prob.topk(5)
    for rank,(score,index) in enumerate(zip(scores.tolist(),indices.tolist()),1):
        rows.append({'model':name,'rank':rank,'label':weights.meta['categories'][index],'probability':score})
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
display(pd.DataFrame(rows));save_result('06_top5',rows)

## Step 4 — Complete the dataset transfer task
Continue with Tutorial 7: it replaces the final classifier, freezes the pretrained base, trains on CIFAR-10 and fine-tunes selected layers. This is different from running an unchanged ImageNet classifier. To use your own photo, change IMAGE_PATH; do not infer ground-truth accuracy from one unlabelled image.

References: [Torchvision models and weights](https://docs.pytorch.org/vision/stable/models.html).

## Explain it in class
Describe the input, target, model, loss, update rule and evaluation split. Use the displayed results from your own run. Changing the dataset, seed or training budget can change the result.

**Course source:** the matching tutorial PDF supplied in the Deep Learning course. Original explanatory implementation; the course PDFs are not redistributed.